# 01 · Exploración de datos — BankNextAction AI

Cuaderno de **exploración** (no es código de producción). La carga, la partición y la figura viven en `src/banknextaction/`; aquí solo se usan y se comentan. Requiere haber corrido `uv run python scripts/prepare_data.py`.

In [ ]:
from banknextaction.config import load_config
from banknextaction.data.load import load_raw, make_target
from banknextaction.data.split import chronological_boundaries, split_summary

cfg = load_config("../configs/config.yaml")
raw = load_raw(cfg.path(cfg.data.raw_file), cfg.data.separator)
y = make_target(raw, cfg.data)
raw.shape, raw.dtypes.value_counts()

## Variables disponibles antes del contacto
Se usan solo variables conocidas **antes** de llamar al cliente. `duration`, `campaign`, `contact`, `day` y `month` se excluyen (ver `configs/config.yaml` y `docs/DECISIONS.md` D-002).

In [ ]:
raw[list(cfg.features.numeric)].describe().round(1)

In [ ]:
for col in cfg.features.categorical:
    print(col, raw[col].value_counts().to_dict())

## La tasa de contratación cambia con el tiempo
El archivo está ordenado por fecha (mayo 2008 – noviembre 2010). La prevalencia de `y = yes` sube de forma marcada al final del periodo, así que cada conjunto tiene su propio piso.

In [ ]:
bounds = chronological_boundaries(len(raw), cfg.split.train_frac, cfg.split.val_frac)
split_summary(y, bounds)

![EDA](../reports/figures/01_eda.png)

**Lectura:** el resultado de la campaña previa (`poutcome`) es la señal más fuerte; `pdays == -1` y `poutcome == 'unknown'` marcan a quien nunca fue contactado (82 % de las filas).